# TEN Kaggle T4 Smoke Test

Use this notebook on Kaggle with GPU accelerator set to T4. It loads Kaggle limit order book data, builds the TEN spoofing detector, runs a short training smoke test, and saves a small result report.

In [ ]:
# Section 1: Kaggle runtime check
import os
import sys
import json
import time
from pathlib import Path

import torch

print('Python:', sys.version.split('\n')[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('/kaggle/input exists:', Path('/kaggle/input').exists())
print('/kaggle/working exists:', Path('/kaggle/working').exists())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Section 2: Find repo root and prepare imports
import sys
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path('/kaggle/working'),
    Path('/kaggle/input/solana-ai-trading-system'),
    Path('/kaggle/input/Solana-AI-Trading-System'),
]
repo_root = None
for candidate in candidate_roots:
    if (candidate / 'services' / 'models' / 'spoofing_detection' / 'src').exists():
        repo_root = candidate / 'services' / 'models' / 'spoofing_detection'
        break
    if (candidate / 'src').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError('Could not find the spoofing_detection repo root. Attach the code or run from the project folder.')

src_path = repo_root / 'src'
sys.path.insert(0, str(src_path))
print('Repo root:', repo_root)
print('Source path added:', src_path)

In [ ]:
# Section 3: Import core TEN modules
import numpy as np
from data.kaggle_loader import prepare_kaggle_data
from data.dataset import create_data_splits, create_dataloaders
from models.ten import TENSpoofingDetector
from training.trainer import TENTrainer, TrainingConfig

print('TEN modules imported successfully')

In [ ]:
# Section 4: Load Kaggle data
asset = 'BTC'
frequency = '1min'
max_rows = 20000

kaggle_data_dir = None
for candidate in [
    repo_root / 'data' / 'raw' / 'kaggle_crypto',
    Path('/kaggle/input/kaggle-crypto-limit-order-book-data'),
    Path('/kaggle/input/high-frequency-crypto-limit-order-book-data'),
    Path('/kaggle/input'),
]:
    if candidate.exists():
        kaggle_data_dir = candidate
        break

if kaggle_data_dir is None:
    raise FileNotFoundError('Could not find Kaggle crypto data. Attach the dataset in Kaggle and re-run.')

print('Data directory:', kaggle_data_dir)
X, y = prepare_kaggle_data(
    asset=asset,
    frequency=frequency,
    max_rows=max_rows,
    data_dir=kaggle_data_dir,
)
print('X shape:', X.shape)
print('Positive rate:', float(y.mean()))

In [ ]:
# Section 5: Split data and build loaders
seed = 42
datasets = create_data_splits(X, y, seed=seed)
loaders = create_dataloaders(datasets, batch_size=32, num_workers=0)

for name, ds in datasets.items():
    print(name, len(ds), 'samples', 'positive=', ds.n_positive)

print('Train batches:', len(loaders['train']))
print('Val batches:', len(loaders['val']))
print('Test batches:', len(loaders['test']))

In [ ]:
# Section 6: Build TEN model and trainer
input_dim = X.shape[2]
model = TENSpoofingDetector(
    d_input=input_dim,
    d_model=256,
    n_heads=8,
    d_ff=1024,
    n_layers=6,
    n_classes=2,
)

config = TrainingConfig(
    batch_size=32,
    max_epochs=3,
    lr=5e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    patience=2,
    checkpoint_dir=str(repo_root / 'checkpoints' / f'kaggle_ten_{asset.lower()}_{frequency}'),
    log_file=str(repo_root / f'training_log_kaggle_ten_{asset.lower()}_{frequency}.csv'),
    device='auto',
)

trainer = TENTrainer(model, config)
print('Model parameters:', sum(p.numel() for p in model.parameters()))
print('Device:', config.resolve_device())

In [ ]:
# Section 7: Run a short smoke training
best = trainer.train(loaders['train'], loaders['val'])
trainer.load_best_checkpoint()
test_metrics = trainer.evaluate(loaders['test'])

print('Best validation metrics:', best)
print('Test metrics:', test_metrics)

In [ ]:
# Section 8: Save test results
results = {
    'asset': asset,
    'frequency': frequency,
    'data_shape': tuple(X.shape),
    'positive_rate': float(y.mean()),
    'best_val_f1': float(best.get('val_f1', 0.0)),
    'best_val_precision': float(best.get('val_precision', 0.0)),
    'best_val_recall': float(best.get('val_recall', 0.0)),
    'test_f1': float(test_metrics.get('f1', 0.0)),
    'test_precision': float(test_metrics.get('precision', 0.0)),
    'test_recall': float(test_metrics.get('recall', 0.0)),
}

output_path = repo_root / 'kaggle_ten_smoke_test_results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print('Saved results to', output_path)
print(results)